# SoundStream training on Kaggle

**Before running:**
1. Add dataset [LibriSpeech](https://www.kaggle.com/datasets/a24998667/librispeech) (or update `LIBRI_INPUT` below).
2. Add Kaggle secrets:
   - `GITHUB_TOKEN` - if the repo is private
   - `WANDB_API_KEY` - for W&B logging
3. Enable GPU.

In [1]:
import os
import subprocess
import sys
from pathlib import Path
import shutil
import time
import torch

REPO_DIR = "soundstream_hw"
REPO_URL = "https://github.com/ndrew1337/soundstream_hw.git"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = f"https://{token}@github.com/ndrew1337/soundstream_hw.git"
except Exception:
    pass

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
if str(Path(REPO_DIR).resolve()) not in sys.path:
    sys.path.insert(0, str(Path(REPO_DIR).resolve()))

Cloning into 'soundstream_hw'...


/kaggle/working/soundstream_hw


In [2]:
os.environ["PIP_NO_WARN_CONFLICTS"] = "1"
!pip install -q torchmetrics pystoi "numba>=0.59" librosa soundfile requests tqdm wget matplotlib pandas wandb hydra-core omegaconf

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 4.1 MB/s eta 0:00:00


In [3]:
LIBRI_INPUT = "/kaggle/input/datasets/a24998667/librispeech"

os.makedirs("/kaggle/working/lsdata", exist_ok=True)
for part in ["train-clean-100", "test-clean"]:
    part_src = f"{LIBRI_INPUT}/{part}"
    part_dst = f"/kaggle/working/lsdata/{part}"
    if not os.path.exists(part_dst):
        os.symlink(part_src, part_dst)
!ls /kaggle/working/lsdata/

test-clean  train-clean-100


In [4]:
N_EPOCHS = 120
RUN_NAME = f"v5-kaggle-{N_EPOCHS}ep"
CKPT = Path(f"saved/{RUN_NAME}") / "model_best.pth"

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    pass

import wandb
wandb.login()

print(f"Epochs: {N_EPOCHS}")
print(f"Run: {RUN_NAME}")
print(f"Checkpoint: {CKPT}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: andgri200 (andgri200-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epochs: 120
Run: v5-kaggle-120ep
Checkpoint: saved/v5-kaggle-120ep/model_best.pth


## Train

In [5]:
t0 = time.time()
!HYDRA_FULL_ERROR=1 python train.py -cn baseline_kaggle \
    datasets.train.data_dir=/kaggle/working/lsdata \
    datasets.test.data_dir=/kaggle/working/lsdata \
    trainer.n_epochs={N_EPOCHS} \
    writer.run_name={RUN_NAME} \
    writer.mode=online \
    trainer.override=True
print(f"Elapsed: {(time.time() - t0) / 3600:.2f} h")

Logging git commit and patch...
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: andgri200 (andgri200-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ setting up run ykd8p0lb4n72ljyektuq65zg5zlfe9h2 (0.2s)
wandb: ⣾ setting up run ykd8p0lb4n72ljyektuq65zg5zlfe9h2 (0.2s)
wandb: ⣷ setting up run ykd8p0lb4n72ljyektuq65zg5zlfe9h2 (0.2s)
wandb: ⣯ setting up run ykd8p0lb4n72ljyektuq65zg5zlfe9h2 (0.2s)
wandb: ⣟ setting up run ykd8p0lb4n72ljyektuq65zg5zlfe9h2 (0.2s)
wandb: ⡿ setting up run ykd8p0lb4n72ljyektuq65zg5zlfe9h2 (0.7s)
wandb: ⢿ setting up run ykd8p0lb4n72ljyektuq65zg5zlfe9h2 (0.7s)
wandb: ⣻ setting up run ykd8p0lb4n72ljyektuq65zg5zlfe9h2 (0.7s)
wandb: ⣽ setting up run ykd8p0lb4n72ljyektuq65zg5zlfe9h2 (0.7s)
wandb: ⣾ setting up run ykd8p0lb4n72ljyektuq65zg5zlfe9h2 (0.7s)
wandb: ⣷ setting up run ykd8p

## Inference 

In [6]:
if not Path(CKPT).is_file():
    raise FileNotFoundError(f"Train first or set CKPT. Missing: {CKPT}")

!HYDRA_FULL_ERROR=1 python inference.py \
    datasets.test.data_dir=/kaggle/working/lsdata \
    inferencer.from_pretrained={CKPT} \
    inferencer.save_path=test-clean-final \
    inferencer.save_audio=false \
    inferencer.save_visuals_count=0

[2026-05-20 16:42:56,214][src.datasets.base_audio_dataset][INFO] - Filtered 9 (0.3%) records by audio length [None, 30.0] sec.
SoundStream(
  (encoder): Encoder(
    (conv1): CausalConv1d(
      (conv): Conv1d(1, 32, kernel_size=(7,), stride=(1,))
    )
    (encoder_blocks): Sequential(
      (0): EncoderBlock(
        (elu): ELU(alpha=1.0)
        (res_unit1): ResidualUnit(
          (elu): ELU(alpha=1.0)
          (conv1): CausalConv1d(
            (conv): Conv1d(32, 32, kernel_size=(7,), stride=(1,))
          )
          (conv2): CausalConv1d(
            (conv): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
          )
        )
        (res_unit2): ResidualUnit(
          (elu): ELU(alpha=1.0)
          (conv1): CausalConv1d(
            (conv): Conv1d(32, 32, kernel_size=(7,), stride=(1,), dilation=(3,))
          )
          (conv2): CausalConv1d(
            (conv): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
          )
        )
        (res_unit3): ResidualUnit(
         

In [7]:
ckpt_src = Path(CKPT)
if not ckpt_src.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_src.resolve()}")

out_dir = Path("/kaggle/working/output")
out_dir.mkdir(parents=True, exist_ok=True)
ckpt_dst = out_dir / f"checkpoint-epoch{N_EPOCHS}.pth"
shutil.copy2(ckpt_src, ckpt_dst)
print(f"Saved to {ckpt_dst} ({ckpt_dst.stat().st_size / 1e6:.1f} MB)")

Saved to /kaggle/working/output/checkpoint-epoch120.pth (371.5 MB)
